# Linear Regression

This notebook accompanies the **ML Viz** lesson on linear regression.
We'll implement OLS, Ridge, and Lasso from scratch.

**Companion lesson:** https://ml-viz-ruby.vercel.app/courses/linear-regression/01-linear-regression

> **To save your work:** click the **Copy to Drive** button at the top, or go to File → Save a copy in Drive. Changes to this view are not saved.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams['figure.facecolor'] = '#0f1117'
plt.rcParams['axes.facecolor'] = '#1a1d27'
plt.rcParams['text.color'] = 'white'
plt.rcParams['axes.labelcolor'] = '#94a3b8'
plt.rcParams['xtick.color'] = '#94a3b8'
plt.rcParams['ytick.color'] = '#94a3b8'
plt.rcParams['axes.edgecolor'] = '#2e3347'

## Intuition — the simplest useful model

**Linear regression** fits a straight line (or hyperplane) through data: `ŷ = wx + b`, choosing
`w, b` to minimize the **squared error**. It's the "hello world" of ML, but it earns its keep — it
has a **closed-form solution** (the normal equations, no iteration needed), it's fully interpretable
(each weight is a feature's effect), and it's the foundation everything else generalizes from.
Adding a penalty on the weights gives **Ridge** (L2, shrinks coefficients) and **Lasso** (L1, drives
some to exactly zero for feature selection). We derive OLS from scratch, confirm gradient descent
finds the same answer, and validate against `sklearn`.

## OLS: Closed-Form Solution

$$w^* = (X^T X)^{-1} X^T y$$

In [ ]:
np.random.seed(42)
n = 100
X = 2 * np.random.randn(n, 1) + 1
y = 3.5 * X.squeeze() + 1.2 + 0.5 * np.random.randn(n)

X_b = np.c_[np.ones(n), X]  # add bias column -> shape (n, 2)

# --- Closed-form normal equation: w = (X^T X)^-1 X^T y ---
w_ols = np.linalg.inv(X_b.T @ X_b) @ X_b.T @ y
print("Closed-form:    b = {:.3f}, w = {:.3f}".format(w_ols[0], w_ols[1]))

# --- Gradient descent on the SAME MSE loss ---
# grad of (1/n)||Xw - y||^2  =  (2/n) X^T (Xw - y)
def gradient_descent(X, y, lr=0.05, n_iter=2000):
    w = np.zeros(X.shape[1])
    m = X.shape[0]
    for _ in range(n_iter):
        grad = (2.0 / m) * X.T @ (X @ w - y)
        w = w - lr * grad
    return w

w_gd = gradient_descent(X_b, y)
print("Gradient desc.: b = {:.3f}, w = {:.3f}".format(w_gd[0], w_gd[1]))

# --- Residuals and R^2 ---
y_hat = X_b @ w_ols
resid = y - y_hat
ss_res = np.sum(resid ** 2)
ss_tot = np.sum((y - y.mean()) ** 2)
r2 = 1 - ss_res / ss_tot
print("R^2 = {:.4f}".format(r2))

# --- Plot: fit (left) and residuals (right) ---
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

ax1.scatter(X, y, c='#818cf8', s=15, alpha=0.6, label='Data')
x_line = np.linspace(X.min(), X.max(), 100)
ax1.plot(x_line, w_ols[0] + w_ols[1] * x_line,
         color='#14b8a6', linewidth=2, label='OLS fit')
ax1.legend()
ax1.set_title('Ordinary Least Squares (R^2 = {:.3f})'.format(r2), color='white')
ax1.set_xlabel('x'); ax1.set_ylabel('y')

ax2.scatter(y_hat, resid, c='#f59e0b', s=15, alpha=0.6)
ax2.axhline(0, color='#94a3b8', linewidth=1, linestyle='--')
ax2.set_title('Residuals vs. fitted', color='white')
ax2.set_xlabel('predicted y'); ax2.set_ylabel('residual (y - y_hat)')

plt.tight_layout()
plt.show()

**What to notice:** the closed-form normal equations and gradient descent recover the **same**
`(b, w) ≈ (1.2, 3.5)` — the true data-generating line — and `R² ≈ 0.98`. The right-hand residual
plot is **structureless** (random scatter around 0), which is the sign of a well-specified linear
fit; a pattern there would mean the model is missing something.

## Ridge vs Lasso

Ridge (L2): $\mathcal{L} = \|y - Xw\|^2 + \lambda\|w\|^2$
Lasso (L1): $\mathcal{L} = \|y - Xw\|^2 + \lambda\|w\|_1$

In [ ]:
# Ridge closed form: w = (X^T X + lam*I)^-1 X^T y
def ridge(X, y, lam):
    p = X.shape[1]
    return np.linalg.inv(X.T @ X + lam * np.eye(p)) @ X.T @ y

# Lasso via coordinate descent with soft-thresholding
def soft_threshold(rho, lam):
    if rho > lam:
        return rho - lam
    if rho < -lam:
        return rho + lam
    return 0.0

def lasso_coordinate_descent(X, y, lam, n_iter=500):
    m, p = X.shape
    w = np.zeros(p)
    for _ in range(n_iter):
        for j in range(p):
            # residual excluding feature j's current contribution
            r_j = y - X @ w + X[:, j] * w[j]
            rho = X[:, j] @ r_j / m
            z_j = X[:, j] @ X[:, j] / m
            w[j] = soft_threshold(rho, lam) / z_j
    return w

# Show Ridge shrinking the fit as lambda grows
lambdas = [0.01, 0.1, 1.0, 5.0]
fig, axes = plt.subplots(1, len(lambdas), figsize=(4 * len(lambdas), 4), sharey=True)
fig.suptitle('Ridge Regression: Effect of lambda', color='white', fontsize=13, y=1.02)

for ax, lam in zip(axes, lambdas):
    w_r = ridge(X_b, y, lam)
    ax.scatter(X, y, c='#818cf8', s=10, alpha=0.4)
    ax.plot(x_line, w_r[0] + w_r[1] * x_line, color='#14b8a6', linewidth=2)
    ax.set_title('lambda = {}'.format(lam), color='white', fontsize=11)
plt.tight_layout()
plt.show()

**What to notice:** as the Ridge penalty `λ` grows, the fitted line **flattens** — the slope is
pulled toward 0. Regularization trades a little training fit for smaller, more stable coefficients
that generalize better, especially when data is scarce or features are correlated.

## The library way — validate OLS and Ridge against `sklearn`

Our from-scratch normal equations should match `sklearn.LinearRegression` exactly (both solve the
same least-squares problem), and our closed-form Ridge should match `sklearn.Ridge`. The cell
asserts both.

In [ ]:
from sklearn.linear_model import LinearRegression, Ridge as SkRidge

lin = LinearRegression().fit(X, y)                       # sklearn on the raw (X, y)
print(f'OLS  ours: b={w_ols[0]:.4f}, w={w_ols[1]:.4f}')
print(f'OLS  sklearn: b={lin.intercept_:.4f}, w={lin.coef_[0]:.4f}')
assert np.allclose([w_ols[0], w_ols[1]], [lin.intercept_, lin.coef_[0]]), "OLS must match sklearn"

lam = 1.0
sk_ridge = SkRidge(alpha=lam, fit_intercept=False).fit(X_b, y)   # penalize the bias column too
assert np.allclose(ridge(X_b, y, lam), sk_ridge.coef_), "closed-form Ridge must match sklearn"
print('\nour OLS == sklearn LinearRegression, our Ridge == sklearn Ridge ✓')

**What to notice:** identical coefficients — our hand-derived normal equations *are* what
`sklearn` computes. For linear regression the closed form is exact and fast; you'd only fall back to
gradient descent when the feature matrix is too large to invert.

## Worked example: OLS by hand

From the lesson: $X = [[1,1],[1,2],[1,3]]$, $y = [2,3,5]$. Solve $w^* = (X^\top X)^{-1} X^\top y$.

In [ ]:
X = np.array([[1, 1], [1, 2], [1, 3]])
y = np.array([2, 3, 5])

# Normal equation, by hand the answer is [0.333, 1.5]
w = np.linalg.inv(X.T @ X) @ X.T @ y
print("w* =", w.round(3), "  (intercept, slope)")

# Residuals and R^2 (matches the lesson's hand computation)
y_hat = X @ w
resid = y - y_hat
ss_res = np.sum(resid ** 2)
ss_tot = np.sum((y - y.mean()) ** 2)
print("residuals =", resid.round(3))
print("SS_res = {:.3f}, SS_tot = {:.3f}".format(ss_res, ss_tot))
print("R^2 = {:.3f}".format(1 - ss_res / ss_tot))
print("prediction at x=4:", round(np.array([1, 4]) @ w, 3))

**What to notice:** the 3-point worked example makes the matrix algebra concrete — `(XᵀX)⁻¹Xᵀy`
by hand reproduces the least-squares line. This is the same computation, just small enough to trace
every entry.

## Regularization paths: Ridge vs Lasso

As the penalty $\alpha$ grows, Ridge shrinks weights smoothly toward zero; Lasso drives some to **exactly** zero (feature selection).

In [ ]:
from sklearn.linear_model import Ridge, Lasso
from sklearn.datasets import make_regression

Xr, yr = make_regression(n_samples=100, n_features=8, n_informative=3,
                         noise=10, random_state=0)
alphas = np.logspace(-2, 2, 30)
ridge = np.array([Ridge(a).fit(Xr, yr).coef_ for a in alphas])
lasso = np.array([Lasso(a).fit(Xr, yr).coef_ for a in alphas])

fig, ax = plt.subplots(1, 2, figsize=(13, 4))
ax[0].plot(alphas, ridge); ax[0].set_xscale('log'); ax[0].set_title('Ridge (L2)')
ax[1].plot(alphas, lasso); ax[1].set_xscale('log'); ax[1].set_title('Lasso (L1)')
for a in ax: a.set_xlabel('alpha'); a.set_ylabel('coefficient')
plt.tight_layout(); plt.show()

**What to notice:** the two regularization paths differ in a crucial way. **Ridge (L2)** shrinks
every coefficient smoothly toward 0 but rarely *to* 0. **Lasso (L1)** drives many coefficients to
**exactly 0** as `α` grows — automatic **feature selection**. That corner-seeking behavior is why
Lasso yields sparse, interpretable models.

## Gotchas & tradeoffs

- **Multicollinearity breaks OLS.** Highly correlated features make `XᵀX` nearly singular, so the
  coefficients become huge and unstable (tiny data changes flip them). Ridge fixes this by adding
  `λI` — its original motivation.
- **Regularization needs scaled features.** L1/L2 penalize by coefficient magnitude, so features on
  larger scales get penalized less; **standardize first**.
- **OLS assumes a lot** — linearity, independent homoscedastic (constant-variance) errors. Check the
  residual plot; funnel shapes or curves signal violations.
- **L1 (sparse) vs L2 (shrink).** Lasso selects features; Ridge keeps all but small — Elastic Net
  blends them.

In [ ]:
# Multicollinearity: two nearly-identical features make OLS coefficients explode
np.random.seed(0)
x1 = np.random.randn(60)
x2 = x1 + 1e-3 * np.random.randn(60)          # almost a copy of x1
Xc = np.c_[np.ones(60), x1, x2]
yc = 2.0 * x1 + 0.1 * np.random.randn(60)     # truth only uses x1

w_ols_c   = np.linalg.lstsq(Xc, yc, rcond=None)[0]
w_ridge_c = np.linalg.inv(Xc.T @ Xc + 1.0 * np.eye(3)) @ Xc.T @ yc
print('OLS coefs   (unstable):', w_ols_c.round(2))
print('Ridge coefs (stable)  :', w_ridge_c.round(2))

**What to notice:** with two near-duplicate features, OLS splits the weight between them in a
wild, arbitrary way (large opposing coefficients), while Ridge keeps both modest and stable. This
instability under correlated features is the classic reason to reach for regularization.

## Key takeaways

- Linear regression fits $y = Xw$ by minimizing squared error.
- **OLS** has a closed form: $w^* = (X^\top X)^{-1} X^\top y$.
- **Ridge (L2)** shrinks weights smoothly; **Lasso (L1)** zeros some out (sparse models).
- Regularization trades a little bias for much lower variance — key against overfitting.

---
## ✏️ Your turn

The cells below are **exercise scaffolds**: the concept is recapped, the code outline is set, and `# TODO(you)` marks what you fill in. Run the `assert` cell after each — it passes silently when your answer is right.

### Exercise 1 — OLS via the normal equations

The least-squares weights solve the **normal equations**:

$$X^\top X \, \mathbf{w} = X^\top \mathbf{y}$$

Implement the fit with `np.linalg.solve` (never invert explicitly — solving is faster and numerically safer). The checks recover a perfect line and verify the property the whole method is named for: the residuals are **orthogonal to every column** of $X$.

In [ ]:
def ols_fit(X, y):
    """Least-squares weights via the normal equations."""
    X = np.asarray(X, dtype=float)
    y = np.asarray(y, dtype=float)

    # TODO(you): solve (X^T X) w = X^T y  (hint: np.linalg.solve)
    return ...

In [ ]:
# Checks — run me
x = np.array([0.0, 1.0, 2.0, 3.0])
X = np.column_stack([np.ones_like(x), x])
y = 2 * x + 1
assert np.allclose(ols_fit(X, y), [1, 2]), "perfect line y = 2x + 1 -> intercept 1, slope 2"

rng = np.random.default_rng(0)
Xr = np.column_stack([np.ones(50), rng.standard_normal((50, 2))])
yr = Xr @ np.array([0.5, -1.0, 2.0]) + 0.1 * rng.standard_normal(50)
wr = ols_fit(Xr, yr)
assert np.allclose(Xr.T @ (yr - Xr @ wr), 0, atol=1e-9), "residuals must be orthogonal to every column of X"
assert np.allclose(wr, np.linalg.lstsq(Xr, yr, rcond=None)[0]), "must match np.linalg.lstsq"
# Edge case: near-singular design matrix (two nearly-collinear columns)
rng2 = np.random.default_rng(7)
x1 = rng2.standard_normal(30)
X_collinear = np.column_stack([np.ones(30), x1, x1 + 1e-6 * rng2.standard_normal(30)])
y_collinear = 2.0 + 3.0 * x1 + 0.05 * rng2.standard_normal(30)
w_collinear = ols_fit(X_collinear, y_collinear)
assert np.all(np.isfinite(w_collinear)), "must stay finite even when X^T X is nearly singular"
w_ls_collinear = np.linalg.lstsq(X_collinear, y_collinear, rcond=None)[0]
assert np.allclose(X_collinear @ w_collinear, X_collinear @ w_ls_collinear, atol=1e-2), \
    "predictions must still match lstsq even though the two collinear coefficients individually are unstable"

print("✅ Exercise 1 passed")

<details>
<summary>💡 Show solution</summary>

```python
def ols_fit(X, y):
    X = np.asarray(X, dtype=float)
    y = np.asarray(y, dtype=float)
    return np.linalg.solve(X.T @ X, X.T @ y)
```

</details>

### Exercise 2 — Ridge regression in closed form

Adding the $L_2$ penalty $\lambda \lVert \mathbf{w} \rVert^2$ only changes the normal equations by a diagonal nudge:

$$(X^\top X + \lambda I) \, \mathbf{w} = X^\top \mathbf{y}$$

The checks confirm the three behaviors from the lesson: $\lambda = 0$ recovers OLS, moderate $\lambda$ **shrinks** the weights, and $\lambda \to \infty$ crushes them to zero.

In [ ]:
def ridge_fit(X, y, lam):
    """Ridge weights: solve (X^T X + lam * I) w = X^T y."""
    X = np.asarray(X, dtype=float)
    y = np.asarray(y, dtype=float)
    d = X.shape[1]

    # TODO(you): the regularized normal equations (hint: np.eye(d))
    return ...

In [ ]:
# Checks — run me
assert np.allclose(ridge_fit(Xr, yr, 0.0), wr), "lambda = 0 recovers OLS"

w10 = ridge_fit(Xr, yr, 10.0)
assert np.linalg.norm(w10) < np.linalg.norm(wr), "the penalty shrinks the weights"

w_huge = ridge_fit(Xr, yr, 1e8)
assert np.linalg.norm(w_huge) < 1e-3, "lambda -> infinity crushes the weights to ~0"
print("✅ Exercise 2 passed")

<details>
<summary>💡 Show solution</summary>

```python
def ridge_fit(X, y, lam):
    X = np.asarray(X, dtype=float)
    y = np.asarray(y, dtype=float)
    d = X.shape[1]
    return np.linalg.solve(X.T @ X + lam * np.eye(d), X.T @ y)
```

</details>

---
## 🌐 Extra practice — from Open-Deep-ML

Three quick [DML](https://github.com/Open-Deep-ML/DML-OpenProblem)-style drills that complement the OLS/Ridge/Lasso work above: the normal equation as its own reusable function, feature scaling as data prep, and a small bank of regression metrics.

### Exercise 3 — Normal equation as a standalone function (DML #14)

Package the closed-form solve from section 1 into DML's exact signature: a function that takes plain lists (not just NumPy arrays) and returns coefficients rounded to **four** decimal places. The checks cross-check it against a fresh gradient-descent fit on the same toy data (both should land on the same line, mirroring section 1's OLS-vs-GD comparison) and probe a near-singular design matrix — two nearly-identical columns, the classic failure mode for the normal equation.

In [ ]:
def linear_regression_normal_equation(X, y):
    """DML #14 signature: lists in, list out, rounded to 4 decimals."""
    X = np.asarray(X, dtype=float)
    y = np.asarray(y, dtype=float).reshape(-1, 1)

    # TODO(you): theta = (X^T X)^-1 X^T y, then round to 4 decimals and flatten to a list
    return ...

In [ ]:
# Checks — run me
assert np.allclose(linear_regression_normal_equation([[1, 1], [1, 2], [1, 3]], [1, 2, 3]), [0.0, 1.0]), \
    "DML's own example: perfect line y = 1*x + 0"

# Cross-check against gradient descent (section 1's `gradient_descent`) on a fresh toy dataset
rng3 = np.random.default_rng(3)
x3 = 2 * rng3.standard_normal(80) + 1
y3 = 3.5 * x3 + 1.2 + 0.5 * rng3.standard_normal(80)
X3_b = np.c_[np.ones(80), x3]

theta_closed = linear_regression_normal_equation(X3_b.tolist(), y3.tolist())
theta_gd = gradient_descent(X3_b, y3, lr=0.05, n_iter=2000)
assert np.allclose(theta_closed, theta_gd, atol=0.05), \
    "closed-form and gradient descent must converge to (nearly) the same line"

# Edge case: near-singular design matrix (two nearly-collinear columns)
rng4 = np.random.default_rng(7)
x4 = rng4.standard_normal(30)
X_collinear2 = np.column_stack([np.ones(30), x4, x4 + 1e-6 * rng4.standard_normal(30)])
y_collinear2 = 2.0 + 3.0 * x4 + 0.05 * rng4.standard_normal(30)
theta_collinear = linear_regression_normal_equation(X_collinear2.tolist(), y_collinear2.tolist())
assert np.all(np.isfinite(theta_collinear)), "must stay finite even when X^T X is nearly singular"
pred = X_collinear2 @ np.array(theta_collinear)
pred_ls = X_collinear2 @ np.linalg.lstsq(X_collinear2, y_collinear2, rcond=None)[0]
assert np.allclose(pred, pred_ls, atol=1e-2), \
    "predictions should still match lstsq despite unstable individual coefficients"
print("✅ Exercise 3 passed")

<details>
<summary>💡 Show solution</summary>

```python
def linear_regression_normal_equation(X, y):
    X = np.asarray(X, dtype=float)
    y = np.asarray(y, dtype=float).reshape(-1, 1)
    theta = np.linalg.inv(X.T @ X) @ X.T @ y
    return np.round(theta, 4).flatten().tolist()
```

</details>

### Exercise 4 — Feature scaling (DML #16)

Gradient descent (and many regularized fits) converges faster and more evenly when features share a common scale. Implement both **standardization** ($z$-score) and **min-max normalization** in one function. The checks match DML's example exactly and probe a zero-variance column (every row identical) — the classic divide-by-zero trap.

In [ ]:
def feature_scaling(data):
    """Return (standardized, normalized) versions of `data`, each rounded to 4 decimals."""
    data = np.asarray(data, dtype=float)

    # TODO(you): standardize -> (data - mean) / std   (guard std == 0 -> treat as 1)
    standardized = ...

    # TODO(you): min-max normalize -> (data - min) / (max - min)  (guard range == 0 -> treat as 1)
    normalized = ...

    return np.round(standardized, 4).tolist(), np.round(normalized, 4).tolist()

In [ ]:
# Checks — run me
std_out, norm_out = feature_scaling(np.array([[1, 2], [3, 4], [5, 6]]))
assert std_out == [[-1.2247, -1.2247], [0.0, 0.0], [1.2247, 1.2247]], "DML's standardization example"
assert norm_out == [[0.0, 0.0], [0.5, 0.5], [1.0, 1.0]], "DML's min-max example"

# Edge case: a zero-variance (constant) column must not produce NaN/Inf
data_const = np.array([[5.0, 1.0], [5.0, 2.0], [5.0, 3.0]])
std_const, norm_const = feature_scaling(data_const)
std_const, norm_const = np.array(std_const), np.array(norm_const)
assert np.all(np.isfinite(std_const)) and np.all(np.isfinite(norm_const)), \
    "a constant column (std=0, range=0) must not divide by zero into NaN/Inf"
assert np.allclose(std_const[:, 0], 0.0) and np.allclose(norm_const[:, 0], 0.0), \
    "a constant column has no meaningful deviation from itself -> scaled to 0"
print("✅ Exercise 4 passed")

<details>
<summary>💡 Show solution</summary>

```python
def feature_scaling(data):
    data = np.asarray(data, dtype=float)
    mean, std = data.mean(axis=0), data.std(axis=0)
    std_safe = np.where(std == 0, 1.0, std)
    standardized = (data - mean) / std_safe

    mn, mx = data.min(axis=0), data.max(axis=0)
    range_safe = np.where(mx - mn == 0, 1.0, mx - mn)
    normalized = (data - mn) / range_safe

    return np.round(standardized, 4).tolist(), np.round(normalized, 4).tolist()
```

</details>

### Exercise 5 — Regression metrics bank (DML #69, #71, #93)

Three small, frequently-asked metrics — grouped into one bank since each is a one-liner around the residuals `y_true - y_pred`:

- **R²** (#69): fraction of variance explained, $1 - \text{SS}_{res}/\text{SS}_{tot}$
- **RMSE** (#71): $\sqrt{\text{mean}((y_{true}-y_{pred})^2)}$
- **MAE** (#93): $\text{mean}(|y_{true}-y_{pred}|)$

The checks match DML's examples and probe the classic R² edge case: a **constant** `y_true` (zero variance) makes $\text{SS}_{tot} = 0$, which would otherwise divide by zero.

In [ ]:
def r_squared(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    if np.array_equal(y_true, y_pred):
        return 1.0
    ss_tot = np.sum((y_true - y_true.mean()) ** 2)
    ss_res = np.sum((y_true - y_pred) ** 2)
    # TODO(you): guard ss_tot == 0 -> return 0.0 (constant y_true, imperfect fit)
    # otherwise: return round(1 - ss_res / ss_tot, 3)
    return ...


def rmse(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    if y_true.shape != y_pred.shape or y_true.size == 0:
        raise ValueError("y_true and y_pred must be non-empty arrays of the same shape")
    # TODO(you): round(sqrt(mean((y_true - y_pred)**2)), 3)
    return ...


def mae(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    if y_true.shape != y_pred.shape or y_true.size == 0:
        raise ValueError("y_true and y_pred must be non-empty arrays of the same shape")
    # TODO(you): round(mean(abs(y_true - y_pred)), 3)
    return ...

In [ ]:
# Checks — run me
assert r_squared(np.array([1, 2, 3, 4, 5]), np.array([1.1, 2.1, 2.9, 4.2, 4.8])) == 0.989, "DML #69 example"
assert rmse(np.array([3, -0.5, 2, 7]), np.array([2.5, 0.0, 2, 8])) == 0.612, "DML #71 example"
assert mae(np.array([3, -0.5, 2, 7]), np.array([2.5, 0.0, 2, 8])) == 0.5, "DML #93 example"

# Perfect predictions
assert r_squared(np.array([1.0, 2.0, 3.0]), np.array([1.0, 2.0, 3.0])) == 1.0
assert rmse(np.array([1.0, 2.0, 3.0]), np.array([1.0, 2.0, 3.0])) == 0.0
assert mae(np.array([1.0, 2.0, 3.0]), np.array([1.0, 2.0, 3.0])) == 0.0

# Edge case: constant y_true (zero variance) -> SS_tot = 0
y_const = np.array([4.0, 4.0, 4.0])
assert r_squared(y_const, y_const) == 1.0, "matching a constant target exactly -> R^2 = 1"
assert r_squared(y_const, np.array([4.0, 5.0, 3.0])) == 0.0, \
    "SS_tot = 0 with an imperfect fit is undefined variance-explained -> defined as 0.0, not a crash"
print("✅ Exercise 5 passed")

<details>
<summary>💡 Show solution</summary>

```python
def r_squared(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    if np.array_equal(y_true, y_pred):
        return 1.0
    ss_tot = np.sum((y_true - y_true.mean()) ** 2)
    ss_res = np.sum((y_true - y_pred) ** 2)
    if ss_tot == 0:
        return 0.0
    return round(1 - ss_res / ss_tot, 3)


def rmse(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    if y_true.shape != y_pred.shape or y_true.size == 0:
        raise ValueError("y_true and y_pred must be non-empty arrays of the same shape")
    return round(float(np.sqrt(np.mean((y_true - y_pred) ** 2))), 3)


def mae(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    if y_true.shape != y_pred.shape or y_true.size == 0:
        raise ValueError("y_true and y_pred must be non-empty arrays of the same shape")
    return round(float(np.mean(np.abs(y_true - y_pred))), 3)
```

</details>